[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sapiosciences/sapio-py-tutorials/blob/master/02_query_data_records.ipynb)

# Create Connection

Every notebook starts by creating a *SapioUser*. That object holds the webservice URL and the credentials for the request.

How to authenticate is covered in [API Authentication](01_api_authentication.ipynb). An API user from User Manager signs in with a username and password. An API key from the Webservice API page grants admin access. A bearer token can be used when the system has OAuth configured. The connection below uses an API user.

`verify_ssl_cert` is `False` here because the example server uses a self-signed certificate. Leave it set to `True` when connecting to a server with a valid SSL/TLS certificate.

In [ ]:
from sapiopylib.rest.User import SapioUser
from sapiopylib.rest.DataRecordManagerService import DataRecord
from sapiopylib.rest.DataRecordManagerService import DataRecordPojoPageCriteria
from sapiopylib.rest.DataMgmtService import DataMgmtServer
from sapiopylib.rest.DataRecordManagerService import DataRecordPojoHierarchyPageCriteria

user = SapioUser(url="https://linux-vm:8443/webservice/api",
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7", account_name="sapio",
                 username="pyRestTest", password="Password1!", verify_ssl_cert=False)
dataRecordManager = DataMgmtServer.get_data_record_manager(user)

These commands do not produce any results yet.

Now let's query for the record with record ID 1.

# Query Data Records

In [ ]:
# Get all Directory records whose ids are in the list [1] (so just a single record with id = 1)
result = dataRecordManager.query_data_records_by_id("Directory", [1])
if len(result.result_list) > 0:
    result_list: list = result.result_list
    element: DataRecord = result_list[0]
    print(element.get_record_id())
    print(element.get_fields())

In the example above, we query for the directory whose record ID is 1, which happens to be the root directory. We print the record ID and the fields of the record.
Alternatively, you can print a `DataRecordPojo` object or a page of results directly to inspect the data record names.

In [ ]:
# Get all Samples with SampleId in the list ["000007"]
result = dataRecordManager.query_data_records("Sample", "SampleId", ["000007"])
print(result)

Paging is available. In the next example, we get samples 10 per page, for 3 pages.

In [ ]:
next_page = DataRecordPojoPageCriteria(page_size=10)
for i in range(3):
    result = dataRecordManager.query_all_records_of_type("Sample", next_page)
    next_page = result.next_page_criteria
    print("Page " + str(i + 1) + ": " + str(result))
    if not result.is_next_page_available:
        break

`query_system_for_record` returns the `DataRecordPojo` if it finds the record, or `None` if the record is not found.

In [ ]:
result = dataRecordManager.query_system_for_record('Directory', 1)
print(result)
result = dataRecordManager.query_system_for_record('Directory', 1000000)
print(result)

You can query both the parents and children of records.
The simplest form retrieves the parent of a single record.

In this example, we retrieve an FCS file record and look up its parent sample.

In [ ]:
fcs_result = dataRecordManager.query_all_records_of_type("FCSFile", DataRecordPojoPageCriteria(page_size=1))
if len(fcs_result.result_list) > 0:
    fcs_record = fcs_result.result_list[0]
    parents = dataRecordManager.get_parents(fcs_record.get_record_id(), "FCSFile", "Sample")
    if len(parents.result_list) > 0:
        parent_sample = parents.result_list[0]
        print(parent_sample)

In a slightly more complex case, we use a batch call to retrieve the parents of multiple FCS records at once.

A batch call is often much faster than making a call for each record in a loop, because it reduces HTTP and remote overhead.

The result object is slightly different. To demonstrate paging in this situation, we request pages of 10 parents, for the first five pages of parents of the first 100 FCS records.
Note: Each FCS record has a parent, so this does not complete the entire query. The remaining pages are never computed or returned.

In [ ]:
fcs_result = dataRecordManager.query_all_records_of_type("FCSFile", DataRecordPojoPageCriteria(page_size=100))
samples = list()
if len(fcs_result.result_list) > 0:
    fcs_record_id_list = [x.get_record_id() for x in fcs_result.result_list]

    fcs_record_by_record_id = dict((x.get_record_id(), x) for x in fcs_result.result_list)

    next_parent_page_criteria = DataRecordPojoHierarchyPageCriteria(page_size=10)
    for i in range(5):
        print("Page " + str(i + 1))

        parents = dataRecordManager.get_parents_list(fcs_record_id_list, "FCSFile", "Sample", next_parent_page_criteria)
        next_parent_page_criteria = parents.next_page_criteria

        for source_record_id, result_records in parents.result_map.items():
            if result_records is None or len(result_records) == 0:
                continue
            
            samples.extend(result_records)

            fcs_record = fcs_record_by_record_id.get(source_record_id)
            if fcs_record is not None:
                print(str(fcs_record) + " -> [" + ', '.join([str(x) for x in result_records]) + "]")
                
        if not parents.is_next_page_available:
            break

You can convert a list of data records into a pandas DataFrame, which can be used by external tools.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 100)
samples_data_frame = dataRecordManager.get_data_frame(samples)
display(samples_data_frame)

# Auto-Paging

Recall that in the previous sections of the tutorial we used a paging loop like this:


```
next_page = DataRecordPojoPageCriteria(page_size=10)
for i in range(3):
    result = dataRecordManager.query_all_records_of_type("Sample", next_page)
    next_page = result.next_page_criteria
    print("Page " + str(i + 1) + ": " + str(result))
    if not result.is_next_page_available:
        break
```

You will often need to page through records in this fashion. You may not care which page you are on, and just want all of the records (or all of the records up to a limit that you choose).

In this case, the auto-paging API can save quite a bit of work:

In [ ]:
from sapiopylib.rest.utils.autopaging import *

pager = QueryAllRecordsOfTypeAutoPager("Directory", user, DataRecordPojoPageCriteria(page_size=10))
for record in pager:
  print(str(record))

In the example above, all auto-paging classes from `sapiopylib` can be imported from the `sapiopylib.rest.utils.autopaging` module.
Each auto-pager class also implements its own iterator. An auto-pager is a one-time iterator that cannot go backward. If you want to revisit elements, store the records you have obtained in another container, such as a list or a file.

It is also possible to obtain all results in a single call. For classes such as `QueryAllRecordsOfTypeAutoPager`, the difference between iterating in a loop and obtaining all results at once is easy to see: the all-at-once call simply collects the results into a single list.
So the code above has the same effect as the code below:

In [ ]:
pager = QueryAllRecordsOfTypeAutoPager("Directory", user, DataRecordPojoPageCriteria(page_size=3))
all_records: List[DataRecord] = pager.get_all_at_once()

for record in all_records:
  print(str(record))

For other pagers, such as hierarchical pagers and side link pagers, there are subtle differences between the two approaches that you need to remember.

Consider the following code:

In [ ]:
children_pager = GetChildrenListAutoPager([x.record_id for x in all_records], "Directory",
                                                  user, DataRecordPojoHierarchyPageCriteria(page_size=3))

The children_pager variable is also an iterator. This time, however, it is not an iterator of data records. Instead, it is a **partial** dictionary that maps each record ID to its child records (of data type "Directory", in this case).
What does it mean for the dictionary to be partial? It means that:


1.   A single occurrence of a record ID key does not necessarily contain all of that record's children.
2.   The complete list of children for a given record ID is the union of all results keyed by that record ID.

This is the same structure that is returned by the data record manager and defined in the Sapio REST API specification. A partial result occurs when a page ends in the middle of a record's children, which can happen when the number of children does not divide evenly into the page size.


Therefore, to correctly collect all of the children for all of the provided record IDs, we must combine the results from every page:

In [ ]:
# A Multimap can have the same key multiple times with different values.
# In the code below, all_results is a SetMultimap which maps IDs (int) to sets of child DataRecords.
# It can be thought of as a dict structured like:
# {
#   1: {DataRecord 1, DataRecord 2},
#   2: {DataRecord 3, DataRecord 4},
#   1: {DataRecord 5, DataRecord 6}
# }
# 
# Adding a second entry with 1 as the key does not replace the first entry.
all_results: SetMultimap[int, DataRecord] = SetMultimap()
for parent_record_id, children_records_partial in children_pager:
  all_results.put_all(parent_record_id, children_records_partial)

It would be **incorrect** to replace the dictionary value for each parent_record_id, because that would discard the partial results from earlier pages.

However, you can avoid the work of collecting the partial results yourself by obtaining all of the results at once, as long as the entire result fits in your runtime's memory.

The code above is therefore equivalent to:

In [ ]:
all_results: SetMultimap[int, DataRecord] = children_pager.get_all_at_once()

Note that using record models implies that you have already decided to fit all pages into memory at once.